# Facies Classification from Well Log Data
## Machine learning for energy applications — Module 1, Exercise 4

In this exercise we build a supervised classification model to predict geological **facies** from well log measurements. Facies classification is a core task in subsurface characterization: petrophysicists and geologists use wireline log responses (gamma ray, resistivity, porosity, photoelectric effect, etc.) to identify rock types encountered during drilling.

The dataset originates from a carbonate reservoir study in the Hugoton gas field, Kansas. Nine distinct facies have been labelled by an expert interpreter. Our goal is to train a Random Forest classifier that can reproduce those labels given the log curves alone, **in a well it has never seen**.

### Learning objectives
1. Load, merge and explore a real well-log dataset, and audit its data quality well by well.
2. Visualize feature distributions and class balance.
3. Prepare data — apply fixed cleaning rules and engineer features before the split; keep fitted steps (imputation, encoding) inside a pipeline.
4. Hold out one whole well as the test set and compare a Random Forest with a majority-class baseline on cross-validation folds made of whole wells.
5. Tune hyperparameters with GridSearchCV on the same well-grouped folds.
6. Open the test well once, for the locked model, and interpret the model through feature importances and a confusion matrix.

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

print("Libraries loaded successfully.")

---
## 2. Load and Explore the Data

We have two CSV files:
- **facies_dataset.csv** — 4,149 depth samples from 10 wells, each with wireline log measurements and an expert-assigned facies label. Depths are in metres (as the `Description` column states), with samples nominally 0.5 m apart within a well; confirm the units with the data owner before building depth-derived features.
- **facies_labels.csv** — a lookup table mapping the integer facies code (1–9) to a descriptive rock-type name.

After loading we merge them so that every sample carries both the numeric code and the human-readable facies name.

In [ ]:
# Load datasets
df = pd.read_csv("facies_dataset.csv")
labels = pd.read_csv("facies_labels.csv")

print(f"Dataset shape : {df.shape}")
print(f"Labels shape  : {labels.shape}")
print()
print("Columns:", list(df.columns))
df.head()

In [ ]:
# Merge facies names into the main dataframe
df = df.merge(labels, on="Facies", how="left")

print("Facies codes and names (the label encoding used throughout):")
print(labels.to_string(index=False))
print()
df.head()

In [ ]:
# Summary statistics and missing values
print("=== Descriptive Statistics ===")
display(df.describe())

print("\n=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found.")

print("\n=== Wells in Dataset ===")
print(df["Well Name"].unique())

### 2.1 Audit by Well

Missing values and implausible readings are rarely spread evenly: they come with particular wells, tools or logging runs. Before any modelling, count them **per well**, and compare each curve's maximum with its median. PE (photoelectric factor) above 20 and GR above 200 API deserve a question to the data owner.

In [ ]:
# Data-quality audit, one row per well
by_well = df.groupby("Well Name", sort=False)
audit = pd.DataFrame({
    "rows": by_well.size(),
    "GR missing %": by_well["GR"].apply(lambda s: 100 * s.isna().mean()).round(1),
    "PE missing %": by_well["PE"].apply(lambda s: 100 * s.isna().mean()).round(1),
    "GR > 200 API": by_well["GR"].apply(lambda s: int((s > 200).sum())),
    "PE > 20": by_well["PE"].apply(lambda s: int((s > 20).sum())),
    "depths sorted": by_well["Depth"].apply(lambda s: s.is_monotonic_increasing),
    "repeated depths": by_well["Depth"].apply(lambda s: int(s.duplicated().sum())),
})
display(audit)

for col in ["GR", "PE"]:
    print(f"{col}: median {df[col].median():.2f}, maximum {df[col].max():.2f}, "
          f"missing in {df[col].isna().sum()} of {len(df)} rows")

---
## 3. Visualize the Data

Before modelling it is important to understand class balance and inter-feature relationships. In well-log interpretation, certain facies are inherently rarer (e.g., phylloid-algal bafflestone), which affects classifier performance.

These plots describe all ten wells, including the one held out as the test well in section 6. That is acceptable for describing the data, but no modelling choice below is taken from them; in a project, set the test well aside before looking at its labels.

In [ ]:
# Facies distribution bar chart
facies_order = labels["Facies_type"].tolist()
counts = df["Facies_type"].value_counts().reindex(facies_order)

fig, ax = plt.subplots(figsize=(10, 5))
counts.plot(kind="bar", color=sns.color_palette("Set2", n_colors=9), edgecolor="k", ax=ax)
ax.set_title("Distribution of Facies Types", fontsize=14)
ax.set_xlabel("Facies")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot of key wireline features coloured by facies
key_features = ["GR", "ILD_log10", "DeltaPHI", "PHIND"]
pair_df = df[key_features + ["Facies_type"]].dropna()

g = sns.pairplot(
    pair_df,
    hue="Facies_type",
    palette="Set2",
    diag_kind="kde",
    plot_kws={"alpha": 0.4, "s": 15},
    height=2.2,
)
g.figure.suptitle("Pair Plot of Key Well Log Features", y=1.02, fontsize=14)
plt.show()

---
## 4. Data Preparation

We need to:
1. **Drop non-predictive columns** — `Well Name` is an identifier, not a feature (we keep a copy as `Well` for the train/test split and the grouped folds). The `Description` column is auto-generated text that repeats the well name, depth, formation and GR: it carries no additional signal and would expose the well identity. `Facies_type` is the label in words.
2. **Apply fixed cleaning rules** — PE values above 20 are recording errors in this file (every other PE value is below 8.1), so we set them to missing. A fixed rule learns nothing from the data, so it may run before the split. GR above 200 API can be real in radioactive shales; we keep those values and flag them for the data owner.
3. **Impute missing values inside the pipeline** — Median imputation learns medians, so it is fitted later, inside the model pipeline, on training wells only. PE is missing in two whole wells, so the imputer also adds missing-value indicator columns: the model can tell an imputed PE from a measured one.
4. **Encode categorical features inside the pipeline** — `Formation` is a geological zone name. `OrdinalEncoder` gives each zone an integer code; it learns the list of zones, so it is fitted inside the pipeline too. Integer codes are acceptable for a tree-based model like the Random Forest, which can isolate any code with a few splits, but not for distance-based models such as KNN, which would read the arbitrary codes as distances; those need one-hot encoding.

In [ ]:
# Keep a copy of Well Name for the split and the grouped folds
df["Well"] = df["Well Name"].copy()

# Drop the identifier, the auto-generated text (it repeats the well name) and the label in words
df = df.drop(columns=["Well Name", "Description", "Facies_type"])

# Fixed cleaning rule: learns nothing from the data, so it may run before the split
pe_error = df["PE"] > 20
df.loc[pe_error, "PE"] = np.nan
print(f"PE values above 20 set to missing: {pe_error.sum()}")

# Imputation and Formation encoding are fitted steps: they run inside the pipeline
# (section 7), fitted on training wells only. Missing values left for the pipeline:
missing = df.isnull().sum()
print(missing[missing > 0])

df.head()

---
## 5. Feature Engineering

Domain knowledge can improve model performance:
- **GR bins** — Gamma ray is often interpreted in broad ranges (low = clean sand, high = shale). We discretize it into bins with fixed edges at 45, 75 and 150 API.
- **Depth-normalized features** — Normalizing GR and PHIND by depth can capture trends related to burial compaction. A ratio to depth changes with the depth unit (metres here), which is one more reason to confirm units.

Both are row-by-row formulas with fixed constants, so computing them before the split does not leak information between wells. Missing GR stays missing in the new columns and is imputed inside the pipeline.

In [ ]:
# GR bins with fixed edges (petrophysical convention: low, medium, high, very high gamma ray)
df["GR_bin"] = pd.cut(
    df["GR"],
    bins=[-np.inf, 45, 75, 150, np.inf],
    labels=[0, 1, 2, 3],
).astype(float)

# Depth-normalized features (row-wise formulas; nothing is fitted)
df["GR_depth_norm"] = df["GR"] / df["Depth"]
df["PHIND_depth_norm"] = df["PHIND"] / df["Depth"]

print("New features added: GR_bin, GR_depth_norm, PHIND_depth_norm")
print(f"Data frame now has {df.shape[1]} columns.")
df.head()

---
## 6. Train / Test Split — Leave-One-Well-Out

In subsurface ML a random train/test split would leak spatial information because adjacent depth samples in the same well are highly correlated. Instead we hold out an entire well for testing. This is the **leave-one-well-out** strategy and gives a much more realistic estimate of how the model will perform on a new, unseen well.

We use the **SHANKLE** well as our test set. It stays closed until section 10: every choice before that (the baseline comparison and the tuning) uses the nine training wells only, with cross-validation folds made of whole wells.

In [ ]:
TEST_WELL = "SHANKLE"

# Define feature columns (everything except target and well identifier)
target_col = "Facies"
exclude_cols = [target_col, "Well"]
feature_cols = [c for c in df.columns if c not in exclude_cols]
categorical_cols = ["Formation"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# Split by well
train_mask = df["Well"] != TEST_WELL
test_mask = df["Well"] == TEST_WELL

X_train = df.loc[train_mask, feature_cols]
y_train = df.loc[train_mask, target_col].to_numpy()
groups_train = df.loc[train_mask, "Well"].to_numpy()  # the well of each training row
X_test = df.loc[test_mask, feature_cols]
y_test = df.loc[test_mask, target_col].to_numpy()

print(f"Training samples : {len(X_train)}  (wells: {df.loc[train_mask, 'Well'].nunique()})")
print(f"Test samples     : {len(X_test)}  (well: {TEST_WELL}, closed until section 10)")
print(f"Features         : {len(feature_cols)}")
print(f"\nFeature list: {feature_cols}")

---
## 7. Default Random Forest Against a Majority-Class Baseline

Imputation and encoding go into one pipeline with the model, so every fit — in each cross-validation fold and in the final refit — learns its medians and formation codes from its own training wells only.

We compare a default-parameter Random Forest with a **majority-class baseline** (always predict the most common facies) on the same folds. The folds are made of whole training wells (`GroupKFold`, three wells per validation fold), so each fold asks the deployment question: how well does the model do on wells it has not seen? For contrast only, we also score the forest with shuffled row folds, which put neighbouring samples of the same well on both sides of every split. We report macro F1, which weights every facies equally, and accuracy. The test well is not used here.

In [ ]:
# Steps that learn from data (medians, formation codes): fitted inside every pipeline fit
preprocess = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median", add_indicator=True), numeric_cols),
        ("formation", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
         categorical_cols),
    ],
    verbose_feature_names_out=False,
)


def make_pipeline_for(model):
    """Preprocessing and model in one pipeline, so each fit refits both."""
    return Pipeline([("prep", clone(preprocess)), ("model", model)])


baseline_clf = make_pipeline_for(DummyClassifier(strategy="most_frequent"))
rf_default = make_pipeline_for(RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))

well_folds = GroupKFold(n_splits=3)  # whole wells: three training wells per validation fold
for i, (_, val_idx) in enumerate(well_folds.split(X_train, y_train, groups_train)):
    print(f"Fold {i}: validation wells {sorted(set(groups_train[val_idx]))}")
row_folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # contrast only

rows = []
for name, model, cv, groups in [
    ("Majority-class baseline, well folds", baseline_clf, well_folds, groups_train),
    ("Default Random Forest, well folds", rf_default, well_folds, groups_train),
    ("Default Random Forest, shuffled row folds (contrast only)", rf_default, row_folds, None),
]:
    res = cross_validate(model, X_train, y_train, groups=groups, cv=cv,
                         scoring=["f1_macro", "accuracy"])
    rows.append({
        "model and folds": name,
        "macro F1": res["test_f1_macro"].mean(),
        "accuracy": res["test_accuracy"].mean(),
        "macro F1 per fold": np.round(res["test_f1_macro"], 3),
    })
display(pd.DataFrame(rows).round(3))

# Fit on all nine training wells, for the importances in section 8 (not scored on the test well)
rf_default.fit(X_train, y_train)

# The Formation codes the encoder learned from the training wells
formation_codes = rf_default.named_steps["prep"].named_transformers_["formation"].categories_[0]
print("Formation encoding:", {name: code for code, name in enumerate(formation_codes)})

---
## 8. Feature Importance

Random Forests provide a built-in measure of feature importance (mean decrease in impurity). This helps petrophysicists understand which log curves the model relies on most — and whether that aligns with domain expectations. These importances describe how the forest splits the training wells; they are not causal effects, and they tend to favour features with many distinct values. Module 2 compares them with permutation importance.

In [ ]:
# Feature importance bar chart (default forest, fitted on the nine training wells)
feature_names = rf_default.named_steps["prep"].get_feature_names_out()
importances = rf_default.named_steps["model"].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind="barh", color="steelblue", edgecolor="k", ax=ax)
ax.set_title("Random Forest Feature Importances", fontsize=14)
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()

---
## 9. Hyperparameter Tuning with GridSearchCV

We search over a grid of key Random Forest parameters:
- **n_estimators** — number of trees (more trees = smoother predictions but longer training).
- **max_depth** — controls how deep each tree can grow (limits overfitting).
- **min_samples_split** — minimum samples required to split a node.

The folds follow wells, as in section 7: `GroupKFold(n_splits=3)` on the nine training wells, with `groups=groups_train` passed to `fit`. A row-level `cv=5` would mix samples of the same well across training and validation folds, so it would not measure performance on unseen wells. We select on macro F1 so that rare facies count. The best cross-validated score is the maximum of 18 noisy numbers, so it is optimistic; compare the differences between settings with the fold-to-fold spread. `GridSearchCV` then refits the chosen settings on all nine training wells. The test well is still closed.

In [ ]:
param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
}

grid_search = GridSearchCV(
    make_pipeline_for(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    param_grid,
    cv=GroupKFold(n_splits=3),  # folds of whole training wells
    scoring="f1_macro",
    verbose=1,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train, groups=groups_train)

best = grid_search.best_index_
fold_scores = [grid_search.cv_results_[f"split{i}_test_score"][best] for i in range(3)]
print(f"\nBest grouped-CV macro F1 : {grid_search.best_score_:.4f}  (folds: {np.round(fold_scores, 3)})")
print(f"Best Parameters          : {grid_search.best_params_}")

---
## 10. Confusion Matrix

Now, and only now, we open the test well: the tuned pipeline is locked, and we score it on SHANKLE once. The confusion matrix reveals which facies pairs the model confuses most. Neighbouring facies with overlapping log responses are the usual confusions in the Hugoton data, for example the nonmarine sandstone and the two nonmarine siltstones, or wackestone and packstone-grainstone. Look for the largest off-diagonal cells in this well.

In [ ]:
# The test well is opened once, here, for the locked (tuned) pipeline
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

# Facies names for the classes present in the test well or its predictions
facies_names = labels.set_index("Facies")["Facies_type"].to_dict()
present_labels = sorted(set(y_test) | set(y_pred_tuned))
target_names = [facies_names[f] for f in present_labels]

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred_tuned, labels=present_labels)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_title(f"Confusion Matrix — Tuned Random Forest, test well {TEST_WELL}", fontsize=14)
ax.set_ylabel("True Facies")
ax.set_xlabel("Predicted Facies")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

---
## 11. Final Evaluation

We summarize the best hyperparameters, the grouped cross-validation score that chose them (a validation estimate, optimistic because it is the best of 18), and the test-well scores of the locked pipeline next to the majority-class baseline on the same rows. One well is one noisy sample of the new-well question: report the score as such, and do not go back and re-tune on it.

In [ ]:
acc_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned, average="macro")

# Majority-class baseline on the same test rows (fitted on the training wells)
y_pred_base = baseline_clf.fit(X_train, y_train).predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base, average="macro")

print("=" * 66)
print("       FINAL MODEL EVALUATION (test well used once)")
print("=" * 66)
print(f"  Best Parameters              : {grid_search.best_params_}")
print(f"  Best grouped-CV macro F1     : {grid_search.best_score_:.4f}  (training wells)")
print(f"  Test well                    : {TEST_WELL}, {len(y_test)} samples")
print(f"  Majority baseline, test well : accuracy {acc_base:.4f}  macro F1 {f1_base:.4f}")
print(f"  Tuned forest, test well      : accuracy {acc_tuned:.4f}  macro F1 {f1_tuned:.4f}")
print("=" * 66)
print()
print(classification_report(y_test, y_pred_tuned, labels=present_labels,
                            target_names=target_names, zero_division=0))

---
## Summary and Next Steps

In this exercise we:
- Loaded and merged well-log data with facies labels from the Hugoton gas field, and audited missing and implausible values well by well.
- Explored class balance and feature relationships through visualizations.
- Applied a fixed cleaning rule and row-wise engineered features before the split, and kept every fitted step (median imputation with missing-value indicators, formation encoding) inside a pipeline.
- Held out one whole well (**leave-one-well-out**) as the test set and compared a default Random Forest with a majority-class baseline on folds made of whole training wells.
- Tuned the forest with GridSearchCV on the same well-grouped folds, then opened the test well once for the locked pipeline.
- Evaluated with accuracy, macro F1, classification reports, and a confusion matrix.

**Ideas for further exploration:**
- Try gradient-boosted trees (XGBoost, LightGBM) — they often outperform Random Forests on tabular data; compare them on the same well-grouped folds.
- Experiment with additional engineered features (e.g., rolling averages of GR over a depth window to capture stratigraphic trends), computed within each well.
- Repeat the whole procedure with each well as the test well in turn (nested cross-validation over wells) to see how much the score varies between wells.
- Address class imbalance with class-weight adjustments, or with resampling such as SMOTE applied inside the training folds only.